# Can physical activity participation information improve predictions of later chronic pain?
## A logistic regression case study using NLSY97 survey data

### Governing Question

How much can including physical activity participation information improve the prediction of chronic pain beyond age and general health alone?

### Indended Audience

Students, researchers, or anyone exploring relationships between general health, physical activity, and later chronic pain. Note that this study is not meant to be used for medical diagnosis. 

### Study Overview

Using data from the National Longitudinal Survey of Youth 1997 (NLSY97), which is a longitudinal study that follows Americans born between 1980-1984, I investigate whether physical activity participation information collected in a 2015-2016 (Round 17) interview helps predict later self-reported chronic pain in a 2019-2020 (Round 19) follow-up.

This case study will use 7 predictor columns: 4 baseline health measurements and 3 physical activity measurements: participation in vigorous, light/moderate, and muscle-strengthening activity.

I compare two logistic regression models: one using age, self-rated health, and work limitation features and another that also includes physical activity participation. A third model, the Dummy Classifier, provides a benchmark for comparison.

| Model | Predictors | Purpose |
|--|--|--|
| Dummy Classifier | None | Establishes the majority-class benchmark |
| Model A: Baseline | Age, self-rated health, and two health-related work limitations | Predicts chronic pain without physical activity information |
| Model B: Extended	| Model A's features plus physical activity participation | Test the added predictive value of including physical activity information |

### Roadmap and Outline

1. Prepare the data: Select eligible respondents, handle missing values in the dataset, and encode categorical variables
2. Split the data: Create training (60%), validation (15%), and test (25%) sets
3. Train the models: Fit the Dummy Classifier and two logistic regression models
4. Evaluate performance: Compare models using balanced accuracy and other classification metrics
5. Interpret results: Determine whether adding physical activity improves prediction and discuss the limitations of this study

## 1. Investigation Contract

### 1.1 Study Design

The following table summarizes the main decisions of this study.

| Concept | Decision |
|---|---|
| Observation unit | One NLSY97 respondent (one row per person) |
| Population of interest | NLSY97 participants with usable information from both the 2015 and 2019 interview rounds |
| Prediction target | Whether a respondent reported chronic pain in 2019 (`1` = Yes, `0` = No) |
| Available information | Baseline Round 17 (2015–2016) age, self-rated health, work limitations, and physical activity participation |
| Learning task| Supervised binary classification: predicting chronic pain or no chronic pain |
| Type of investigation | Predictive and exploratory |
| Algorithm | Logistic regression |
| Value of a useful result | Helps determine whether physical activity participation provides useful predictive information beyond basic health information |
| Consequences of a wrong result | Incorrect predictions may miss respondents reporting pain or incorrectly identify those without pain. Misinterpreting the results can also lead to misleading health conclusions. |

### 1.2 Why Logistic Regression? 

Logistic regression is an algorithm that can estimate the probability of a binary outcome, such as chronic pain or no chronic pain. It is also relatively simple and easy to interpret, allowing me to examine how predictors relate to the estimated probability of chronic pain. Using the same algorithm for both models (Model A: Baseline and Model B: Extended) allows me to directly compare their performance and assess whether adding the additional information can improve prediction.

### 1.3 Non-goals

This study is meant only to investigate the relationship between physical activity, general health information, and chronic pain. It does not attempt to diagnose chronic pain, determine when the pain began, or establish whether physical activity can causes or prevent pain.

## 2. Data Provenance and Fitness for Purpose

### 2.1 Dataset Source

| Item | Description |
|---|---|
| Source | [National Longitudinal Survey of Youth 1997 (NLSY97)](https://www.nlsinfo.org/content/cohorts/nlsy97) |
| Owner | U.S. Bureau of Labor Statistics, National Longitudinal Surveys Program |
| Retrieval | Custom subset downloaded via NLS Investigator on November 14, 2026 |
| Access and Terms | Public-use survey data, follow the [NLS documentation](https://nlsinfo.org/) for citation and redistribution |
| Sample | 8,984 American individuals born between 1980 and 1984, first surveyed in 1997. The original sample also included additional Black and Hispanic respondents to improve representation. |
| Geographic scope | United States |
| Time period | Baseline: 2015 survey (2015–2016, Round 17); Outcome: 2019 survey (2019–2020, Round 19) |

View the dataset card for more information about the NLSY97 dataset: [DATASET_CARD.md](../docs/DATASET_CARD.md)


### 2.2 Variable Definitions

#### Predictor Variables

A. Baseline Health Features

| Variable | Type | Values |
|---|---|---|
| Age (`age_at_interview_2015`) | Numerical | Age in years at the baseline interview |
| General Health (`self_rated_health_2015`) | Ordinal | `1` = Excellent; `2` = Very good; `3` = Good; `4` = Fair; `5` = Poor |
| Health Limits Kind of Work (`health_limits_kind_of_work_2015`) | Binary | `0` = No limitation; `1` = Limitation |
| Health Limits Amount of Work (`health_limits_amount_of_work_2015`) | Binary | `0` = No limitation; `1` = Limitation |

B. Physical Activity Features

| Variable | Type | Values |
|---|---|---|
| Vigorous Activity Status (`vigorous_activity_status_2015`) | Categorical | `0` = Never; `1` = Participates; `2` = Unable |
| Light/Moderate Activity Status (`moderate_activity_status_2015`) | Categorical | `0` = Never; `1` = Participates; `2` = Unable |
| Muscle-Strengthening Activity Status (`strength_activity_status_2015`) | Categorical | `0` = Never; `1` = Participates; `2` = Unable |

#### Target Variable

| Variable | Type | Values |
|---|---|---|
| Chronic Pain (2019) (`chronic_pain_30d_2019`) | Binary target | `0` = No chronic pain; `1` = Chronic pain |

#### Identifier and Metadata Variables

These variables identify respondents or record interview timing. Note that they are not used as predictors.

| Variable | Type | Values |
|---|---|---|
| Respondent ID (`respondent_id`) | Identifier | Unique identifier for each participant |
| Baseline Interview Year (`interview_year_2015_round`) | Metadata | `2015` or `2016` |
| Follow-Up Interview Year (`interview_year_2019_round`) | Metadata | `2019` or `2020` |

#### Note: Special Response Codes

The NLSY97 dataset also uses negative values to identify missing, refused, skipped, or noninterview responses. In this study, these codes are not considered valid measurements of age, health, physical activity, or chronic pain and will be removed during data selection. 

| Code | Meaning |
|---|---|
| `-1` | Refusal |
| `-2` | Don't know |
| `-3` | Invalid skip |
| `-4` | Valid skip |
| `-5` | Noninterview |

### 2.3 Data Limitations and Scope 

This study uses the NLSY97 data to only consider physical activity participation, not exercise frequency or duration. The data also only follows Americans born between 1980 and 1984, meaning respondents are relatively close in age. This limits the model's ability to examine age differences, and the findings may not generalize to younger or older populations. Additionally, missing responses and excluding respondents who did not complete follow-up interviews may also introduce bias.

The following cell loads and inspects the dataset using my local configuration file (`student.toml`):

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dsm_project.config import load_config
from dsm_project.data import load_project_dataset

# Locate and load the config file
CONFIG_PATH = Path("../configs/student.toml")
config = load_config(CONFIG_PATH)

# Load the dataset
dataset = load_project_dataset(config)

# Select classification task
task = dataset.tasks["classification"]

# Get the observations
frame = dataset.observations

# Get the variables from config
ID_COL = dataset.identifier
ALL_FEATURES = list(task.feature_columns)
TARGET = task.target_column

# Define interview year cols
INTERVIEW_YEARS = [
    "interview_year_2015_round",
    "interview_year_2019_round",
]

# Define the four baseline predictors
BASELINE_FEATURES = [
    "age_at_interview_2015",
    "self_rated_health_2015",
    "health_limits_kind_of_work_2015",
    "health_limits_amount_of_work_2015",
]

# Define the three relevant activity predictors
ACTIVITY_FEATURES = [
    "vigorous_activity_status_2015",
    "moderate_activity_status_2015",
    "strength_activity_status_2015",
]

# Group features together
SELECTED_FEATURES = BASELINE_FEATURES + ACTIVITY_FEATURES

# Check that all selected columns exist in the dataset
for col in INTERVIEW_YEARS + SELECTED_FEATURES + [TARGET]:
    assert col in frame.columns, f"Column not found: {col}"

assert ID_COL in frame.columns, "Identifier column is missing"
assert TARGET in frame.columns, "Target column is missing"

# Make a copy of the data (only the relevant columns)
data_subset = frame[[ID_COL] + INTERVIEW_YEARS + SELECTED_FEATURES + [TARGET]].copy()

# Display case study information
print("Dataset:", dataset.provenance.name)
print("Research task: Predict future chronic pain using baseline health and physical activity participation")
print("Target:", TARGET)
print("Num. predictors:", len(SELECTED_FEATURES))
print("Predictors:", SELECTED_FEATURES)

# Display subset information
print("Number of respondents:", len(data_subset))
print("Number of columns:", data_subset.shape[1])

# Preview the subset
display(data_subset.head())

# Check missing values and data types
check = pd.DataFrame({
    "Data type": data_subset[INTERVIEW_YEARS + SELECTED_FEATURES + [TARGET]].dtypes,
    "Missing (NaN)": data_subset[INTERVIEW_YEARS + SELECTED_FEATURES + [TARGET]].isna().sum(),
})
display(check)

Dataset: NLSY97 Physical Activity and Chronic Pain Subset
Research task: Predict future chronic pain using baseline health and physical activity participation
Target: chronic_pain_30d_2019
Num. predictors: 7
Predictors: ['age_at_interview_2015', 'self_rated_health_2015', 'health_limits_kind_of_work_2015', 'health_limits_amount_of_work_2015', 'vigorous_activity_status_2015', 'moderate_activity_status_2015', 'strength_activity_status_2015']
Number of respondents: 8984
Number of columns: 11


,respondent_id,interview_year_2015_round,interview_year_2019_round,age_at_interview_2015,self_rated_health_2015,health_limits_kind_of_work_2015,health_limits_amount_of_work_2015,vigorous_activity_status_2015,moderate_activity_status_2015,strength_activity_status_2015,chronic_pain_30d_2019
0,1,2016,-5,34,3,0,0,1,1,0,-5
1,2,2016,2019,33,2,0,0,0,0,0,0
2,3,-5,2019,-5,-5,-5,-5,-5,-5,-5,0
3,4,2015,2019,34,2,0,0,1,1,0,0
4,5,2015,2019,33,2,0,0,1,1,0,0


,Data type,Missing (NaN)
interview_year_2015_round,int64,0
interview_year_2019_round,int64,0
age_at_interview_2015,int64,0
self_rated_health_2015,int64,0
health_limits_kind_of_work_2015,int64,0
health_limits_amount_of_work_2015,int64,0
vigorous_activity_status_2015,int64,0
moderate_activity_status_2015,int64,0
strength_activity_status_2015,int64,0
chronic_pain_30d_2019,int64,0


We see there are no original missing values in the dataset, and the data types are appropriate for the features and target variable.

## 3. Purposeful exploration

### 3.1 Prepare the eligible cohort

The following cell selects participants who meet the eligibility requirements:
1. Completed both 2015 and 2019 interview rounds
2. Have a valid binary 2019 chronic pain response (`0` or `1`)

The number of respondants remaining at each stage are also displayed below.

In [6]:
# Selecting the valid responses
has_baseline = data_subset["interview_year_2015_round"].isin([2015, 2016])
has_followup = data_subset["interview_year_2019_round"].isin([2019, 2020])
has_valid_target = data_subset["chronic_pain_30d_2019"].isin([0, 1])

# Display counts at each stage
cohort_counts = pd.DataFrame({
    "Selection stage": [
        "Original rows", "Valid baseline interview",
        "Both interview rounds", "Both rounds + valid pain target"
    ],
    "Respondents": [
        len(data_subset), int(has_baseline.sum()),
        int((has_baseline & has_followup).sum()),
        int((has_baseline & has_followup & has_valid_target).sum()),
    ],
})
display(cohort_counts)

# Make a copy of the eligible dataset
eligible_dataset = data_subset.loc[
    has_baseline & has_followup & has_valid_target,
    [ID_COL] + INTERVIEW_YEARS + SELECTED_FEATURES + [TARGET]
].copy()

,Selection stage,Respondents
0,Original rows,8984
1,Valid baseline interview,7103
2,Both interview rounds,6402
3,Both rounds + valid pain target,6389


### 3.2 Clean Feature Values and Finalize Sample

In this section, I replace negative NLSY97 response codes (`-1` through `-5`) with `NaN` (missing values) and verify that the 19 predictor variables contain valid responses.

The following rules are applied:

1. **Age:** Between 30 and 36 years
2. **General health:** Integer values from `1` (Excellent) to `5` (Poor)
3. **Health-related work limitations:** `0` (No) or `1` (Yes)
4. **Physical activity participation:** `0` (Never), `1` (Participates), or `2` (Unable)


In [8]:
# Replace negative codes with NaN
for col in SELECTED_FEATURES:
    eligible_dataset[col] = eligible_dataset[col].mask(eligible_dataset[col] < 0)

# Specify specific columns
AGE_COL = BASELINE_FEATURES[0]
HEALTH_COL = BASELINE_FEATURES[1]
HEALTH_LIMITS_COLS = BASELINE_FEATURES[2:4]

# Cleaning features
eligible_dataset[AGE_COL] = eligible_dataset[AGE_COL].where(
    eligible_dataset[AGE_COL].between(30, 36)
)

eligible_dataset[HEALTH_COL] = eligible_dataset[HEALTH_COL].where(
    eligible_dataset[HEALTH_COL].isin([1, 2, 3, 4, 5])
)

for col in HEALTH_LIMITS_COLS:
    eligible_dataset[col] = eligible_dataset[col].where(
        eligible_dataset[col].isin([0, 1])
    )

for col in ACTIVITY_FEATURES:
    eligible_dataset[col] = eligible_dataset[col].where(
        eligible_dataset[col].isin([0, 1, 2])
    )

# Summary
missing_summary = pd.DataFrame({
    "Invalid/Missing": eligible_dataset[SELECTED_FEATURES].isna().sum(),
    "Percent missing (%)": (
        eligible_dataset[SELECTED_FEATURES].isna().mean() * 100
    ).round(2),
})

display(missing_summary)

,Invalid/Missing,Percent missing (%)
age_at_interview_2015,0,0.00
self_rated_health_2015,8,0.13
health_limits_kind_of_work_2015,18,0.28
health_limits_amount_of_work_2015,18,0.28
vigorous_activity_status_2015,34,0.53
moderate_activity_status_2015,80,1.25
strength_activity_status_2015,37,0.58
